In [5]:
# 6.1: Mapa Interativo de Sinistros em Matão/SP (com estilo Carto Positron livre de bloqueio)
import pandas as pd
import plotly.express as px
import os

# Carregar dados limpos
df = pd.read_csv('../data/matao_limpo.csv', parse_dates=['data'])

# Filtrar coordenadas válidas na região de Matão
df_mapa = df.dropna(subset=['latitude', 'longitude']).copy()
filtro_matao = (
    (df_mapa['latitude'].between(-21.80, -21.40)) & 
    (df_mapa['longitude'].between(-48.60, -48.10))
)
df_mapa = df_mapa[filtro_matao].copy()
df_mapa['data_texto'] = df_mapa['data'].dt.strftime('%d/%m/%Y')

# Ordem e paleta de cores
ordem_gravidade = ['Com óbito', 'Com feridos graves', 'Com feridos leves', 'Sem vítimas']
cores_gravidade = {
    'Com óbito': '#d90429',        # Vermelho vivo
    'Com feridos graves': '#f77f00', # Laranja
    'Com feridos leves': '#e5b124',  # Dourado
    'Sem vítimas': '#2b5c8f'         # Azul
}

# Construir o Mapa usando 'carto-positron'
fig = px.scatter_map(
    df_mapa,
    lat='latitude',
    lon='longitude',
    color='gravidade',
    category_orders={'gravidade': ordem_gravidade},
    color_discrete_map=cores_gravidade,
    hover_name='tipo_sinistro',
    hover_data={
        'latitude': False,
        'longitude': False,
        'gravidade': True,
        'data_texto': True,
        'hora': True,
        'tipo_via': True,
        'logradouro': True,
        'total_vitimas': True
    },
    zoom=12.2,
    center={"lat": -21.6035, "lon": -48.3645},
    map_style='carto-positron',  # << ESTILO PROFISSIONAL E SEM BLOQUEIO 403
    height=680,
    title='Mapeamento dos Sinistros de Trânsito em Matão/SP (2015-2026)'
)

fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(
        title=dict(text='<b>Gravidade</b>'),
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='#cccccc',
        borderwidth=1,
        x=0.01,
        y=0.99
    )
)

# Salvar como página HTML
caminho_html = '../output/graficos/mapa_acidentes.html'
fig.write_html(caminho_html)

print("--- Mapa Atualizado com Sucesso ---")
print(f"Total de sinistros mapeados: {len(df_mapa)}")
print(f"Arquivo interativo salvo em: {caminho_html}")

fig.show()


--- Mapa Atualizado com Sucesso ---
Total de sinistros mapeados: 3648
Arquivo interativo salvo em: ../output/graficos/mapa_acidentes.html


In [3]:
# 6.2: Ranking das Vias Mais Críticas e Exportação do Relatório Anual (Seção 8)

# 1. Ranking das 10 Vias com Mais Sinistros em Matão
ranking_vias = df.groupby('logradouro').agg(
    total_sinistros=('id_sinistro', 'count'),
    obitos=('mortos', 'sum'),
    feridos_graves=('feridos_graves', 'sum'),
    feridos_leves=('feridos_leves', 'sum')
).sort_values(by='total_sinistros', ascending=False).head(10)

print("--- TOP 10 VIAS COM MAIS SINISTROS EM MATÃO ---")
print(ranking_vias)

# 2. Resumo Estatístico Anual Consolidado (Conforme Seção 8 do Guia)
resumo_anual = df.groupby('ano').agg(
    total_sinistros=('id_sinistro', 'count'),
    obitos=('mortos', 'sum'),
    feridos_graves=('feridos_graves', 'sum'),
    feridos_leves=('feridos_leves', 'sum'),
    ilesos=('ilesos', 'sum')
).reset_index()

caminho_relatorio = '../output/relatorios/resumo_anual_matao.csv'
os.makedirs('../output/relatorios', exist_ok=True)
resumo_anual.to_csv(caminho_relatorio, index=False)

print(f"\n--- Relatório Anual Salvo com Sucesso em: {caminho_relatorio} ---")
resumo_anual


--- TOP 10 VIAS COM MAIS SINISTROS EM MATÃO ---
                                    total_sinistros  obitos  feridos_graves  \
logradouro                                                                    
RUA SINHARINHA FROTA                            468       2              11   
SP 310                                          247      43              31   
AVENIDA BALDAN                                  222       2              18   
AVENIDA SETE DE SETEMBRO                        182       0               6   
RUA SAO LOURENCO                                170       2              10   
AVENIDA TROLESI                                 166       5              19   
RUA RUI BARBOSA                                 159       0              19   
AVENIDA PADRE NELSON ANTONIO ROMAO              101       1               6   
SP 326                                           91      16              20   
AVENIDA FRANCISCO MASTROPIETRO                   86       7               5   

   

,ano,total_sinistros,obitos,feridos_graves,feridos_leves,ilesos
0,2015,12,12,0,0,0
1,2016,14,15,0,0,0
2,2017,6,6,0,0,0
3,2018,10,12,0,0,0
4,2019,470,18,8,64,33
5,2020,410,7,51,290,220
6,2021,486,7,41,408,313
7,2022,497,15,58,395,304
8,2023,678,13,49,371,319
9,2024,573,17,44,303,265
